# Unir isótopos + DBH/altura de las bases de dinámica forestal

Este notebook parte de 3 archivos:
- `isotopos_concatenado.xlsx` (hoja `Datos_concatenados`): **la base principal** — solo nos interesan estas muestras.
- `Base_unificada_dinamica_forestal.xlsx` (Galeras, Selva Viva, Sumaco).
- `Base_unificada_ForestPlots_Guacamayos_Oyacachi.xlsx` (Guacamayos, Oyacachi).

**Qué hace el script:**
1. Carga las 2 bases de dinámica y se queda **solo** con columnas de identificación (sitio, parcela, árbol, taxonomía) y de **fecha / DBH / altura por censo** — descarta columnas de rasgos/administrativas (dendrómetro, muestras de madera, comentarios, flags de calidad, coordenadas, vouchers, etc.).
2. Concatena esas 2 bases de dinámica en una sola tabla `dinamica_unificada`.
3. Limpia los identificadores de árbol (`treeID`, `new_tree_ID`) y **colapsa filas duplicadas** que representan el mismo árbol (se detectó que la base de dinámica trae, para algunos árboles, una fila con `treeID` y otra fila con `new_tree_ID`, en vez de una sola fila con ambos datos).
4. Detecta un error de arrastre/copiado en el archivo original de isótopos (2 `Sample_ID` con 46 filas cada uno, mismos valores de isótopos pero `new_TreeID` distinto en cada fila) y excluye esas 92 filas del cruce por no ser confiables.
5. Cruza cada muestra de isótopos restante contra la base de dinámica ya limpia, primero por `old_treeID` ↔ `treeID`, y si no encuentra nada, por `new_TreeID` ↔ `new_tree_ID` (dentro del mismo sitio).
6. Trae al lado de cada muestra de isótopos las columnas `date_census*`, `dbh_census*` y `height_census*_m` del árbol correspondiente.
7. Marca en `QC_flag_dinamica` las muestras sin árbol correspondiente en la base de dinámica, con más de un árbol candidato (coincidencia ambigua), o con ID no confiable.
8. Verifica que las muestras de 2006 tengan censo 1 y las de 2025 tengan censo 3, para confirmar que el cruce quedó bien.
9. Guarda todo en un Excel con 3 hojas: `Isotopos_con_DBH` (resultado final), `Sin_match_dinamica` (para revisión manual) y `Dinamica_unificada` (la base de dinámica ya limpia, por si la necesitas aparte).

## 1. Librerías

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

## 2. Funciones auxiliares

`clean_id` normaliza un identificador de árbol a entero, descartando textos no numéricos como `'x'`, `'y'`, `'?'`, `'no ID'`, que aparecen como placeholders en los datos crudos cuando el árbol no tenía un ID claro.

In [2]:
def clean_id(v):
    if pd.isna(v):
        return pd.NA
    s = str(v).strip()
    if s == '' or not re.match(r'^-?\d+(\.\d+)?$', s):
        return pd.NA
    try:
        return int(float(s))
    except ValueError:
        return pd.NA

## 3. Definir qué columnas se conservan de las bases de dinámica

**Se conservan** (identificación + lo que pediste: fechas de medición, DBH y alturas):
- Identificación: `dataset`, `site`, `PlotID`, `Plot_num`, `subplot`, `treeID`, `new_tree_ID`, `family`, `genus`, `species`.
- Por cada censo (1, 2, 3, y los 2 censos extra que solo tiene la base de Guacamayos/Oyacachi): `date_census*`, `dbh_census*`, `height_census*_m`.

**Se descartan** columnas de rasgos, muestreo de madera, comentarios y control de calidad de campo, por ejemplo: `dendrometer`, `Cambio_sp`, `dead_census*`, `leaf_sample_census*`, `wood_sample_census*`, `JH_herbarium_census*`, `comment*`, `crust_thickness`, `wet_length_cm`, `wet_wood_weight`, `dry_wood_weight`, `X_coord`, `Y_coord`, `voucher_code`, `POM_census*_mm`, `Flag1-5_census*`, `HeightBrokenAt_census*`, `Notas_census*`, entre otras — no aportan a fechas/DBH/altura.

Nota: en `Base_unificada_dinamica_forestal.xlsx` la altura del segundo censo se llama `tree_height_2011_m` (porque ese censo fue en 2011); se renombra a `height_census2_m` para que coincida con el nombre usado en la otra base.

In [3]:
CENSUS_COLS = ['date_census1', 'dbh_census1',
               'date_census2', 'dbh_census2', 'height_census2_m',
               'date_census3', 'dbh_census3', 'height_census3_m',
               'date_census_extra1', 'dbh_census_extra1', 'height_census_extra1_m',
               'date_census_extra2', 'dbh_census_extra2', 'height_census_extra2_m']
ID_COLS = ['dataset', 'site', 'PlotID', 'Plot_num', 'subplot', 'treeID', 'new_tree_ID',
           'family', 'genus', 'species']

## 4. Cargar y concatenar las 2 bases de dinámica forestal

In [4]:
d1 = pd.read_excel('Base_unificada_dinamica_forestal.xlsx', sheet_name='Datos_unificados')
d1 = d1.rename(columns={'tree_height_2011_m': 'height_census2_m'})

d2 = pd.read_excel('Base_unificada_ForestPlots_Guacamayos_Oyacachi.xlsx', sheet_name='Datos_unificados')

keep1 = [c for c in ID_COLS + CENSUS_COLS if c in d1.columns]
keep2 = [c for c in ID_COLS + CENSUS_COLS if c in d2.columns]

dinamica = pd.concat([d1[keep1], d2[keep2]], ignore_index=True, sort=False)
print('Filas totales dinamica:', len(dinamica))
dinamica['site'].value_counts()

Filas totales dinamica: 2115


Galeras       636
Sumaco        625
Selva Viva    354
Guacamayos    297
Oyacachi      203
Name: site, dtype: int64

## 5. Limpiar IDs y colapsar duplicados

Se detectó que para algunos árboles la base trae **2 filas** en vez de 1: una con `treeID` (sin `new_tree_ID`) y otra con `new_tree_ID` (sin `treeID` o con el mismo `treeID`), producto de cómo se fueron actualizando los datos entre censos. Para no perder información y no duplicar árboles al cruzar con isótopos, se agrupan por `(site, treeID)` y, para cada columna, se toma el primer valor no nulo dentro del grupo (esto junta en una sola fila los datos que estaban repartidos en 2).

In [5]:
dinamica['treeID_clean'] = dinamica['treeID'].apply(clean_id)
dinamica['new_tree_ID_clean'] = dinamica['new_tree_ID'].apply(clean_id)

def collapse(group):
    out = {}
    for c in group.columns:
        s = group[c].dropna()
        out[c] = s.iloc[0] if len(s) else np.nan
    return pd.Series(out)

mask_has_id = dinamica['treeID_clean'].notna()
with_id = (dinamica[mask_has_id]
           .groupby(['site', 'treeID_clean'], as_index=False, dropna=False)
           .apply(collapse).reset_index(drop=True))
without_id = dinamica[~mask_has_id].reset_index(drop=True)

dinamica_collapsed = pd.concat([with_id, without_id], ignore_index=True, sort=False)
print('Filas antes de colapsar:', len(dinamica), '-> despues:', len(dinamica_collapsed))

Filas antes de colapsar: 2115 -> despues: 2061


## 6. Cargar la base de isótopos (la principal)

Se toma tal cual quedó en `isotopos_concatenado.xlsx` (hoja `Datos_concatenados`), que ya tiene los sitios homogeneizados (`Galeras`, `Selva Viva`, `Sumaco`, `Oyacachi`, `Guacamayos`) — coinciden exactamente con los nombres de sitio de las bases de dinámica, así que no hace falta remapear nombres de sitio.

In [6]:
iso = pd.read_excel('isotopos_concatenado.xlsx', sheet_name='Datos_concatenados')
iso['old_treeID_clean'] = iso['old_treeID'].apply(clean_id)
iso['new_TreeID_clean'] = iso['new_TreeID'].apply(clean_id)
print('Muestras de isotopos:', len(iso))

Muestras de isotopos: 669


## 6.1 Detectar un error de arrastre/copiado en el archivo original de isótopos

Al revisar por qué muestras de 2006 no traían censo 1 (o muestras de 2025 no traían censo 3), se encontró que **dos `Sample_ID`** (`jh-3939` y `jh-p70-3904`, ambos de Selva Viva) aparecen **46 veces cada uno**, siempre con **los mismos valores exactos de isótopos** (mismo `[C]%`, δ13C, etc. — imposible en mediciones reales) pero con un `new_TreeID` **distinto cada vez**. Esto es un error de arrastre/copiado en el Excel original (`SV_cellulose.xlsx`): los valores de una sola medición isotópica quedaron pegados junto a una columna de IDs de árbol que en realidad pertenecía a otra tabla.

**Consecuencia práctica:** al cruzar por `new_TreeID`, estas 92 filas emparejaban cada una con un árbol distinto (y por lo tanto un DBH distinto) de forma esencialmente aleatoria — de ahí que aparecieran muestras de 2006 sin censo 1 y de 2025 sin censo 3: no era un problema del cruce, sino que el `new_TreeID` de origen no era confiable para esas 92 filas.

Estas filas **se excluyen del cruce** (no se les asigna DBH/altura) y quedan marcadas en `QC_flag_dinamica` para que decidas si las corriges, las deduplicas o las descartas.

In [7]:
dupe_new = iso.groupby('Sample_ID')['new_TreeID_clean'].nunique(dropna=True)
dupe_old = iso.groupby('Sample_ID')['old_treeID_clean'].nunique(dropna=True)
bad_samples = set(dupe_new[dupe_new > 1].index) | set(dupe_old[dupe_old > 1].index)
print('Sample_ID con multiples arbol-ID distintos (dato no confiable):', bad_samples)

is_bad = iso['Sample_ID'].isin(bad_samples)
print('Filas afectadas:', is_bad.sum())

# Para estas filas no confiamos en el treeID/new_TreeID -> no se usan para cruzar
iso.loc[is_bad, 'old_treeID_clean'] = pd.NA
iso.loc[is_bad, 'new_TreeID_clean'] = pd.NA

Sample_ID con multiples arbol-ID distintos (dato no confiable): {'jh-p70-3904', 'jh-3939'}
Filas afectadas: 92


## 7. Cruzar cada muestra de isótopos con su árbol en la base de dinámica

Estrategia de cruce, en orden de prioridad, siempre dentro del mismo `Site`:
1. `old_treeID` (isótopos) vs `treeID` (dinámica).
2. Si no hay coincidencia, `new_TreeID` (isótopos) vs `new_tree_ID` (dinámica).

Se guarda además `match_method` (por cuál de las 2 vías se encontró el árbol) y `match_ambiguous` (si había más de un árbol candidato, en cuyo caso se toma el primero y se marca para revisión).

In [8]:
by_treeID, by_newID = {}, {}
for idx, row in dinamica_collapsed.iterrows():
    if pd.notna(row['treeID_clean']):
        by_treeID.setdefault((row['site'], row['treeID_clean']), []).append(idx)
    if pd.notna(row['new_tree_ID_clean']):
        by_newID.setdefault((row['site'], row['new_tree_ID_clean']), []).append(idx)

def find_match(site, old_id, new_id):
    if pd.notna(old_id):
        idxs = by_treeID.get((site, old_id))
        if idxs:
            return idxs[0], 'treeID', len(idxs) > 1
    if pd.notna(new_id):
        idxs = by_newID.get((site, new_id))
        if idxs:
            return idxs[0], 'new_tree_ID', len(idxs) > 1
    return None, None, False

matches = iso.apply(lambda r: find_match(r['Site'], r['old_treeID_clean'], r['new_TreeID_clean']), axis=1)
iso['_match_idx'] = matches.apply(lambda t: t[0])
iso['match_method'] = matches.apply(lambda t: t[1])
iso['match_ambiguous'] = matches.apply(lambda t: t[2])

print(iso['match_method'].value_counts(dropna=False))
print('Sin match:', iso['_match_idx'].isna().sum())
print('Ambiguos:', iso['match_ambiguous'].sum())

treeID         539
None           105
new_tree_ID     25
Name: match_method, dtype: int64
Sin match: 105
Ambiguos: 1


## 8. Traer las columnas de censo (fecha/DBH/altura) a la tabla de isótopos

In [9]:
dyn_cols_to_bring = ['PlotID', 'Plot_num', 'subplot', 'dataset'] + CENSUS_COLS
dyn_subset = dinamica_collapsed[dyn_cols_to_bring].add_prefix('dyn_')

final = iso.join(dyn_subset, on='_match_idx')
final.shape

(669, 50)

## 9. Marcar filas para revisión manual

- **ID no confiable**: la fila es una de las 92 afectadas por el error de arrastre (sección 6.1) — no se intentó cruzar.
- **Sin coincidencia**: la muestra de isótopos no encontró ningún árbol con ese `treeID`/`new_tree_ID` en la base de dinámica — no tendrá DBH/altura.
- **Coincidencia ambigua**: había más de un árbol candidato con el mismo ID en el mismo sitio; se usó el primero, pero conviene revisar cuál es el correcto.

In [10]:
final['QC_flag_dinamica'] = ''
final.loc[is_bad, 'QC_flag_dinamica'] += 'Sample_ID con multiples arbol-ID distintos en el archivo original (dato no confiable, revisar); '
no_match_mask = final['_match_idx'].isna() & ~is_bad
final.loc[no_match_mask, 'QC_flag_dinamica'] += 'Sin coincidencia en base de dinamica (sin DBH/altura); '
final.loc[final['match_ambiguous'], 'QC_flag_dinamica'] += 'Coincidencia ambigua (mas de 1 arbol candidato, se tomo el primero); '

final[final['QC_flag_dinamica'] != ''][['Sample_ID','Site','Plot','old_treeID','new_TreeID','match_method','QC_flag_dinamica']]

,Sample_ID,Site,Plot,old_treeID,new_TreeID,match_method,QC_flag_dinamica
130,p28-8721,Galeras,28,NaN,8721,new_tree_ID,Coincidencia ambigua (mas de 1 arbol candidato...
146,p28-8878,Galeras,28,NaN,8878,None,Sin coincidencia en base de dinamica (sin DBH/...
164,p60-8885,Galeras,60,NaN,8885,None,Sin coincidencia en base de dinamica (sin DBH/...
203,p48-3698,Galeras,48,NaN,3698,None,Sin coincidencia en base de dinamica (sin DBH/...
206,jh-p70-3904,Selva Viva,70,NaN,3604,None,Sample_ID con multiples arbol-ID distintos en ...
...,...,...,...,...,...,...,...
588,p81-8034,Oyacachi,81,NaN,8034,None,Sin coincidencia en base de dinamica (sin DBH/...
589,p81-8056,Oyacachi,81,NaN,8056,None,Sin coincidencia en base de dinamica (sin DBH/...
590,p81-8035,Oyacachi,81,NaN,8035,None,Sin coincidencia en base de dinamica (sin DBH/...
591,p85-8073,Oyacachi,85,NaN,8073,None,Sin coincidencia en base de dinamica (sin DBH/...


## 9.1 Verificación cruzada: año de muestreo vs. censo esperado

Chequeo pedido para confirmar que el cruce quedó bien: las muestras tomadas en **2006** deberían casi todas tener `dyn_dbh_census1`, y las de **2025** deberían casi todas tener `dyn_dbh_census3` (2023/2024/2025). Excluyendo las 92 filas de ID no confiable, esto se cumple casi perfectamente — los pocos casos restantes ya están cubiertos por las categorías de arriba (sin coincidencia, o un hueco genuino de campo donde se registró la fecha del censo pero no se tomó el DBH).

In [11]:
rest = final[~is_bad]
print('Muestras 2006 sin censo1:', rest[(rest['Year'] == 2006) & (rest['dyn_dbh_census1'].isna())].shape[0],
      'de', (rest['Year'] == 2006).sum())
print('Muestras 2025 sin censo3:', rest[(rest['Year'] == 2025) & (rest['dyn_dbh_census3'].isna())].shape[0],
      'de', (rest['Year'] == 2025).sum())

Muestras 2006 sin censo1: 1 de 286
Muestras 2025 sin censo3: 8 de 195


## 10. Reordenar columnas y guardar

In [12]:
final = final.drop(columns=['_match_idx', 'old_treeID_clean', 'new_TreeID_clean'], errors='ignore')

front_cols = ['Site', 'dyn_PlotID', 'Plot', 'dyn_Plot_num', 'dyn_subplot', 'Elevation_m',
              'Sample_ID', 'old_treeID', 'new_TreeID', 'Year', 'Type',
              'family', 'genus', 'species',
              '[C]%', 'δ13C (‰ v.s.V-PDB)', '[C_b]%', 'bulk_δ13C (‰ v.s.V-PDB)',
              '[N]', 'δ15N (‰ v.s. V-PDB)',
              'dyn_date_census1', 'dyn_dbh_census1',
              'dyn_date_census2', 'dyn_dbh_census2', 'dyn_height_census2_m',
              'dyn_date_census3', 'dyn_dbh_census3', 'dyn_height_census3_m',
              'dyn_date_census_extra1', 'dyn_dbh_census_extra1', 'dyn_height_census_extra1_m',
              'dyn_date_census_extra2', 'dyn_dbh_census_extra2', 'dyn_height_census_extra2_m',
              'match_method', 'match_ambiguous', 'QC_flag_dinamica', 'QC_flag']
other_cols = [c for c in final.columns if c not in front_cols]
final = final[front_cols + other_cols]

out_path = 'isotopos_con_DBH.xlsx'
report = final[(final['QC_flag_dinamica'] != '')][
    ['Sample_ID','Site','Plot','old_treeID','new_TreeID','match_method','QC_flag_dinamica']
]

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    final.to_excel(writer, sheet_name='Isotopos_con_DBH', index=False)
    report.to_excel(writer, sheet_name='Sin_match_dinamica', index=False)
    dinamica_collapsed.to_excel(writer, sheet_name='Dinamica_unificada', index=False)

print('Filas finales:', len(final))
print(f'Guardado en: {out_path}')

Filas finales: 669
Guardado en: isotopos_con_DBH.xlsx


## 11. Resumen de lo que debes revisar manualmente

- **92 filas (`jh-3939` y `jh-p70-3904`, Selva Viva) con error de arrastre/copiado en el Excel original**: los mismos valores de isótopos aparecen repetidos 46 veces cada uno junto a un `new_TreeID` distinto cada vez. Se excluyeron del cruce con la base de dinámica (no tienen DBH/altura asignado) porque no se puede confiar en a qué árbol pertenecen realmente. Recomendación: revisar el archivo `SV_cellulose.xlsx` original, identificar cuál es la fila/árbol correcto para cada una de esas 2 muestras, y decidir si el resto son duplicados a eliminar.
- **13 muestras de isótopos sin árbol correspondiente en la base de dinámica** (no tienen DBH/altura): incluyen algunas de Galeras, Selva Viva, Sumaco, Oyacachi y Guacamayos. Puede ser porque el árbol no fue re-medido en los censos de dinámica, o porque el ID no coincide entre bases. Revisar caso por caso en la hoja `Sin_match_dinamica`.
- **1 coincidencia ambigua** (`p28-8721` en Galeras): había más de un árbol con el mismo `new_tree_ID` en la base de dinámica; se tomó el primero, pero conviene confirmar manualmente cuál es el árbol correcto.
- La hoja `Dinamica_unificada` queda disponible por si necesitas la base de dinámica completa (ya limpia y sin columnas de rasgos) para otros análisis.